# Advanced Resources: Stores

## Limitations of `Resource`

Up to this point the models we have looked have have used a basic `simpy.Resource`.  One downside to `Resource` is that it does not allow you to give individual resources attributes or any type of complex behaviour. For many models, this is sufficient, but there may be instances where you need to track and control individual resources. For example, ambulances, or different types of staff. In this notebook we will explore how to add more complex behaviour using the `Store` and `FilterStore` objects provided by `simpy`.

## 1. Imports

In [1]:
import numpy as np
import itertools
import simpy

In [2]:
# to reduce code these classes can be found in distribution.py
# you can also pip `install sim-tools`
from distributions import (
    Exponential, 
)

from sim_utility import set_trace, trace

## 2. Using a `simpy.Store`

A store is simpy a "box" that we can `put` or `get` our own custom resource objects from in a First In First Out (FIFO) basis.

### 2.1 Parameters

In [16]:
NUM_AMBULANCES    = 10
MEAN_SERVICE_TIME = 60.0   # minutes per call
TRAFFIC_INTENSITY = 0.95    # ρ = λ / (c · μ)
RUN_LENGTH        = 1_000 # minutes
RANDOM_SEED       = 42

# exponential IAT and service time so I have used traffic intensity to control IAT
# ρ = λ / (c · μ)  →  mean_interarrival = mean_service / (ρ · c)
MEAN_INTERARRIVAL = MEAN_SERVICE_TIME / (TRAFFIC_INTENSITY * NUM_AMBULANCES)
# → 60 / (0.8 × 10) = 7.5 minutes between calls

### 2.2 Entity classes

In [4]:
class Ambulance:
    """
    An ambulance resource

    Parameters:
    ----------
    ambulance_id: int
        Unique id of the ambulance
    """
    def __init__(self, ambulance_id: int):
        self.ambulance_id = ambulance_id
        self.total_jobs   = 0
        # cumulative busy time
        self.total_busy   = 0.0          
        
    def __repr__(self):
        """
        A text representation of the ambulance to help debugging
        """
        return f"Ambulance(id={self.ambulance_id})"

In [5]:
class Patient:
    """
    A class to hold patient attributes

    Parameters
    ----------
    patient_id: int
        Unique patient id

    arrival_time: float
        Time of arrival to the simulation    
    """
    def __init__(self, patient_id: int, arrival_time: float):
        self.patient_id   = patient_id
        self.arrival_time = arrival_time

    def __repr__(self):
        """
        A text representation of the Patient for debug
        """
        return f"Patient(id={self.patient_id})"

## 2.3 Ambulance dispatch and service process

In [6]:
def dispatch_ambulance(
    env: simpy.Environment,
    store: simpy.Store,
    patient: Patient,
    dists: dict,
    log: dict
) -> None:
    """Simulate ambulance dispatch and service: 
    
    1. queues a patient up for the next free Ambulance (FIFO), 
    2. simulates service (as a single distribution), 
    3. returns the Ambulance to the store.

    Parameters:
    ----------
    env: simpy.Environment
        The simpy environment for the simulation
    store: simpy.Store
        A store of Ambulance objects
    dists: dict
        Contains the "service" distribution
    log: dict
        Audit dictionary
    """

    # Wait for an available Ambulance
    # note we `get()` an Ambulance from the store
    ambulance: Ambulance = yield store.get()

    wait_time = env.now - patient.arrival_time
    log["wait_times"].append(wait_time)

    # Service (travel + on-scene + return)
    service_time = dists["service"].sample()

    # debug
    trace(
        f"{env.now:.1f}  {patient} → {ambulance}  "
        f"(waited {wait_time:.1f} min, service {service_time:.1f} min)"
    )

    yield env.timeout(service_time)

    # Update ambulance stats and return (put) to store
    ambulance.total_jobs += 1
    ambulance.total_busy += service_time
    store.put(ambulance)

    log["service_times"].append(service_time)
    log["assignments"].append((patient.patient_id, ambulance.ambulance_id))

### 2.4 Patient arrival generator

In [7]:
def patient_arrivals_generator(
    env: simpy.Environment,
    store: simpy.Store,
    distributions: dict,
    results: dict
) -> None:
    """
    Arrival process for patients to the ambulance sim.

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    store: simpy.Store
        A store of Ambulance objects

    distributions: dict
        Contains "arrival" and "service" distributions

    results: dict
        Results dictionary
    """
    for patient_id in itertools.count(start=1):

        # time until next patient arrival
        inter_arrival_time = distributions["arrival"].sample()
        yield env.timeout(inter_arrival_time)

        results["n_arrivals"] += 1
        patient = Patient(patient_id, env.now)

        # debug info
        trace(f"{env.now:.1f}: Arrival. {patient}")

        # create ambulance dispatch + service process
        env.process(dispatch_ambulance(env, store, patient, distributions, results))

### 2.5 Single run function

In [17]:
def single_run(
    mean_iat: float = MEAN_INTERARRIVAL,
    mean_service: float = MEAN_SERVICE_TIME,
    n_ambulances: int = NUM_AMBULANCES,
    run_length: float = RUN_LENGTH, 
    random_seed: int = 1
):
    """
    Set up and perform a single replication of the MMS model
    """

    # 1. distribution objects
    dists = {
        "arrival": Exponential(mean_iat, random_seed=1),
        "service": Exponential(mean_service, random_seed=2),
    }

    # 2. simpy environment 
    env = simpy.Environment()

    # 3. Initialise Store
    # 3.1 Create empty Store with sufficient slots
    store = simpy.Store(env, capacity=n_ambulances)

    # 3.2 Create Ambulance objects
    ambulances = [Ambulance(i + 1) for i in range(n_ambulances)]

    # 3.3 `put` Ambulance objects into the store
    for amb in ambulances:
        store.put(amb)

    # 4. results dictionary
    log = {"n_arrivals": 0, "wait_times": [], "service_times": [], "assignments": []}

    env.process(patient_arrivals_generator(env, store, dists, log))
    env.run(until=run_length)

    return ambulances, log

In [20]:
set_trace(False)
# no warm-up.
ambulances, log = single_run(random_seed=42)

waits = np.array(log["wait_times"])
services = np.array(log["service_times"])
n = len(waits)

print("\n" + "═" * 55)
print(f"  Mean inter-arrival   : {MEAN_INTERARRIVAL:.2f} min")
print("─" * 55)
print(f"  Patients arrived     : {log['n_arrivals']}")
print(f"  Patients served      : {n}")
print(f"  Mean wait time       : {waits.mean():.2f} min")
print(f"  P(wait > 0)          : {(waits > 0).mean():.2%}")
print(f"  95th pct wait        : {np.percentile(waits, 95):.2f} min")
print("─" * 55)
print(f"  {'Ambulance':<15} {'Jobs':>6} {'Utilisation':>12}")
print("─" * 55)

# calculate utilisation of each individual ambulance
for amb in ambulances:
    util = amb.total_busy / RUN_LENGTH
    print(f"  Ambulance {amb.ambulance_id:<5}     {amb.total_jobs:>6}       {util:>8.2%}")
print("═" * 55)

Simulation tracing set to: False

═══════════════════════════════════════════════════════
  Mean inter-arrival   : 6.32 min
───────────────────────────────────────────────────────
  Patients arrived     : 149
  Patients served      : 149
  Mean wait time       : 4.97 min
  P(wait > 0)          : 39.60%
  95th pct wait        : 22.91 min
───────────────────────────────────────────────────────
  Ambulance         Jobs  Utilisation
───────────────────────────────────────────────────────
  Ambulance 1             12         74.71%
  Ambulance 2             11         58.18%
  Ambulance 3             13         80.96%
  Ambulance 4             18         77.13%
  Ambulance 5             18         82.55%
  Ambulance 6              8         70.16%
  Ambulance 7             15         71.77%
  Ambulance 8             14         78.69%
  Ambulance 9             10         81.75%
  Ambulance 10            20         69.14%
═══════════════════════════════════════════════════════
